In [1]:
"""
==================================================================================
 CUSTOMER RETENTION INTELLIGENCE PLATFORM - SECONDARY DATASET (Olist)
 Phase 1: Data Understanding -> File 4 of 5: olist_order_payments_dataset.csv
==================================================================================
 Purpose : Explore payment transaction data (used for Tableau dashboard).
           Contains payment method, installments, and value per order.
==================================================================================
"""

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)


def section(title: str) -> None:
    print("\n" + "=" * 90)
    print(f" {title}")
    print("=" * 90)


# ----------------------------------------------------------------------------------
# 1. DATA COLLECTION
# ----------------------------------------------------------------------------------
section("1. DATA COLLECTION")

DATA_PATH = "/Users/meetnakrani/Library/Mobile Documents/com~apple~CloudDocs/Customer-Retention-intelligence-Platform/DataSet/RAW/secondary Data/olist_order_payments_dataset.csv"
df = pd.read_csv(DATA_PATH)

print(f"Dataset loaded successfully from: {DATA_PATH}")
print(f"Shape of dataset -> Rows: {df.shape[0]}, Columns: {df.shape[1]}")


# ----------------------------------------------------------------------------------
# 2. DATASET STRUCTURE
# ----------------------------------------------------------------------------------
section("2. DATASET STRUCTURE (First & Last Records)")
print("\n--- First 5 records ---")
print(df.head())
print("\n--- Last 5 records ---")
print(df.tail())

section("2.1 COLUMN NAMES & DATA TYPES")
print(df.dtypes)

section("2.2 MEMORY USAGE & GENERAL INFO")
df.info(memory_usage="deep")


# ----------------------------------------------------------------------------------
# 3. MISSING VALUE ANALYSIS
# ----------------------------------------------------------------------------------
section("3. MISSING VALUE ANALYSIS")

missing_count = df.isnull().sum()
missing_percent = (missing_count / len(df)) * 100
missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing %": missing_percent.round(2)
})
missing_summary = missing_summary[missing_summary["Missing Count"] > 0].sort_values("Missing %", ascending=False)

if missing_summary.empty:
    print("No missing values found in the dataset.")
else:
    print(missing_summary)


# ----------------------------------------------------------------------------------
# 4. DUPLICATE RECORD CHECK
# ----------------------------------------------------------------------------------
section("4. DUPLICATE RECORD CHECK")
print(f"Fully duplicated rows                          : {df.duplicated().sum()}")
print(f"Duplicated (order_id + payment_sequential) combos: "
      f"{df.duplicated(subset=['order_id', 'payment_sequential']).sum()}")
print("\nNote: multiple rows can share the same order_id when a single order")
print("was paid using more than one payment method/installment sequence.")


# ----------------------------------------------------------------------------------
# 5. PAYMENT TYPE DISTRIBUTION
# ----------------------------------------------------------------------------------
section("5. PAYMENT TYPE DISTRIBUTION")
payment_counts = df["payment_type"].value_counts()
payment_percent = df["payment_type"].value_counts(normalize=True) * 100
payment_summary = pd.DataFrame({"Count": payment_counts, "Percentage": payment_percent.round(2)})
print(payment_summary)


# ----------------------------------------------------------------------------------
# 6. STATISTICAL SUMMARY - INSTALLMENTS & PAYMENT VALUE
# ----------------------------------------------------------------------------------
section("6. STATISTICAL SUMMARY - INSTALLMENTS & PAYMENT VALUE")
print(df[["payment_sequential", "payment_installments", "payment_value"]].describe().T)

section("6.1 INSTALLMENT DISTRIBUTION")
print(df["payment_installments"].value_counts().sort_index())


# ----------------------------------------------------------------------------------
# 7. ZERO / UNUSUAL VALUE CHECK
# ----------------------------------------------------------------------------------
section("7. ZERO / UNUSUAL VALUE CHECK")
zero_value_payments = (df["payment_value"] == 0).sum()
zero_installments = (df["payment_installments"] == 0).sum()
print(f"Payments with payment_value == 0        : {zero_value_payments}")
print(f"Payments with payment_installments == 0 : {zero_installments}")


# ----------------------------------------------------------------------------------
# 8. CARDINALITY CHECK
# ----------------------------------------------------------------------------------
section("8. CARDINALITY CHECK")
print(df.nunique().sort_values())
print(f"\nUnique orders in this file: {df['order_id'].nunique()} "
      f"(vs {df.shape[0]} total payment rows)")


# ----------------------------------------------------------------------------------
# 9. INITIAL DATA QUALITY OBSERVATIONS
# ----------------------------------------------------------------------------------
section("9. INITIAL DATA QUALITY OBSERVATIONS")

observations = f"""
1. Dataset contains {df.shape[0]} payment transactions across {df['order_id'].nunique()}
   unique orders -> some orders have multiple payment rows (split/multi-method payments).
2. No missing values were found in this file.
3. '{payment_counts.index[0]}' is the dominant payment method at {payment_percent.iloc[0].round(1)}%
   of all transactions -> a strong candidate for a Tableau payment-mix chart.
4. {zero_value_payments} row(s) have a payment_value of 0 and {zero_installments} row(s)
   have payment_installments of 0 -> both are worth flagging/investigating as
   potential data quality edge cases (e.g. 'not_defined' payment type rows)
   before they are aggregated into revenue KPIs.
5. Join key to 'olist_orders_dataset.csv' is 'order_id'.
"""
print(observations)

section("ORDER PAYMENTS DATASET - DATA UNDERSTANDING COMPLETE")


 1. DATA COLLECTION
Dataset loaded successfully from: /Users/meetnakrani/Library/Mobile Documents/com~apple~CloudDocs/Customer-Retention-intelligence-Platform/DataSet/RAW/secondary Data/olist_order_payments_dataset.csv
Shape of dataset -> Rows: 103886, Columns: 5

 2. DATASET STRUCTURE (First & Last Records)

--- First 5 records ---
                           order_id  payment_sequential payment_type  payment_installments  payment_value
0  b81ef226f3fe1789b1e8b2acac839d17                   1  credit_card                     8          99.33
1  a9810da82917af2d9aefd1278f1dcfa0                   1  credit_card                     1          24.39
2  25e8ea4e93396b6fa0d3dd708e76c1bd                   1  credit_card                     1          65.71
3  ba78997921bbcdc1373bb41e913ab953                   1  credit_card                     8         107.78
4  42fdf880ba16b47b59251dd489d4441a                   1  credit_card                     2         128.45

--- Last 5 records ---
    